In [ ]:
# This Jupyter Notebook file serves as a cell-by-cell script to generate the results of the project in a single file.

In [ ]:
# All necessary imports. Please run pip install -r requirements.txt to install the required packages if they are missing.
import os
import json
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
import scipy.sparse as sp
import shutil
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import animation
from nltk.corpus import stopwords
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score


In [ ]:
# GENERATE THE 'playlists' SQLITE DATABASE
# This script processes the Spotify Million Playlist Dataset Challenge data and creates a SQLite database with playlists features.

# ----------------------------------------------------------------------------------------------
# NOTE: SKIP THIS CELL IF THE playlists.db HAS ALREADY BEEN MADE—IT TAKES FAIRLY LONG TO RUN
# ----------------------------------------------------------------------------------------------

# --- Config ---
os.makedirs('dbs', exist_ok=True)

# Change this path to the folder containing the 1000 JSON files downloaded and extracted from:
# https://www.aicrowd.com/challenges/spotify-million-playlist-dataset-challenge
DATA_DIR = 'SpotifyData/'  # Folder containing 1000 JSON files

DB_PATH = 'dbs/playlists.db'     # Output SQLite file

# --- Connect to SQLite ---
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

# --- Create Tables ---
cur.executescript("""
DROP TABLE IF EXISTS playlist_track;
DROP TABLE IF EXISTS playlists;
DROP TABLE IF EXISTS tracks;

CREATE TABLE playlists (
    pid INTEGER PRIMARY KEY,
    name TEXT,
    collaborative TEXT,
    modified_at INTEGER,
    num_albums INTEGER,
    num_tracks INTEGER,
    num_followers INTEGER,
    num_edits INTEGER,
    duration_ms INTEGER,
    num_artists INTEGER
);

CREATE TABLE tracks (
    track_uri TEXT PRIMARY KEY,
    artist_name TEXT,
    track_name TEXT,
    album_uri TEXT,
    album_name TEXT,
    artist_uri TEXT,
    duration_ms INTEGER
);

CREATE TABLE playlist_track (
    pid INTEGER,
    track_uri TEXT,
    pos INTEGER,
    PRIMARY KEY (pid, track_uri),
    FOREIGN KEY (pid) REFERENCES playlists(pid),
    FOREIGN KEY (track_uri) REFERENCES tracks(track_uri)
);
""")

# --- Load and Insert Data ---
conn.execute("BEGIN")  # Begin transaction for performance

# Loop through all JSON files
file_count = 0
playlist_count = 0
for filename in os.listdir(DATA_DIR):
    if filename.endswith('.json'):
        file_path = os.path.join(DATA_DIR, filename)
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            for playlist in data['playlists']:
                # Insert into playlists table
                cur.execute("""
                    INSERT OR IGNORE INTO playlists (
                        pid, name, collaborative, modified_at,
                        num_albums, num_tracks, num_followers,
                        num_edits, duration_ms, num_artists
                    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                """, (
                    playlist['pid'],
                    playlist['name'],
                    playlist['collaborative'],
                    playlist['modified_at'],
                    playlist['num_albums'],
                    playlist['num_tracks'],
                    playlist['num_followers'],
                    playlist['num_edits'],
                    playlist['duration_ms'],
                    playlist['num_artists']
                ))

                # Insert each track and link to playlist
                for track in playlist['tracks']:
                    cur.execute("""
                        INSERT OR IGNORE INTO tracks (
                            track_uri, artist_name, track_name,
                            album_uri, album_name, artist_uri, duration_ms
                        ) VALUES (?, ?, ?, ?, ?, ?, ?)
                    """, (
                        track['track_uri'],
                        track['artist_name'],
                        track['track_name'],
                        track['album_uri'],
                        track['album_name'],
                        track['artist_uri'],
                        track['duration_ms']
                    ))

                    # Link playlist to track
                    cur.execute("""
                        INSERT OR IGNORE INTO playlist_track (
                            pid, track_uri, pos
                        ) VALUES (?, ?, ?)
                    """, (
                        playlist['pid'],
                        track['track_uri'],
                        track['pos']
                    ))

                playlist_count += 1
        file_count += 1
        if file_count % 100 == 0:
            print(f"Processed {file_count} files...")

# --- Finalize ---
conn.commit()
conn.close()

print(f"\nDone: {playlist_count} playlists imported from {file_count} files into {DB_PATH}")

In [ ]:
# GENERATES THE 'song_features' SQLITE DATABASE CONTAINING SONG FEATURES FROM THE 'playlists' DATABASE
database_file = 'dbs/playlists.db'
def create_song_feature_db_from(db_file):
    try:
        query = """
                SELECT
                    t.track_uri,
                    t.track_name,
                    t.artist_name,
                    COUNT(pt.pid) AS in_number_of_playlists,
                    AVG(p.num_followers) AS avg_playlist_followers,
                    AVG(pt.pos + 1) AS avg_position_in_playlist
                FROM
                    playlist_track pt
                JOIN
                    playlists p ON pt.pid = p.pid
                JOIN
                    tracks t ON pt.track_uri = t.track_uri
                GROUP BY
                    t.track_uri, t.track_name, t.artist_name
                ORDER BY
                    in_number_of_playlists DESC;
                """
        with sqlite3.connect(db_file) as conn:
            song_features_df = pd.read_sql_query(query, conn)
        new_db = 'song_features.db'
        table_name = 'song_features'
        with sqlite3.connect(new_db) as conn_new:
            song_features_df.to_sql(table_name, conn_new, if_exists='replace', index=False)
            print(f"Song features database created successfully in {new_db} with table '{table_name}'.")
        print(song_features_df.head())
        conn.close()
        conn_new.close()
            
    except Exception as e:
        print(f"Error: {e}")
    
create_song_feature_db_from(database_file)
shutil.move('song_features.db', 'dbs/')
print("Song features database creation completed.")

In [ ]:
# UPDATES 'song_features' DB WITH ENGINEERED FEATURES
conn = sqlite3.connect('dbs/song_features.db')
query = "SELECT * from song_features"

song_df = pd.read_sql_query(query, conn)
conn.close()

#  get standardized stats for our target features
columns_to_standardize = ['in_number_of_playlists', 'avg_playlist_followers', 'avg_position_in_playlist']
mu_std = song_df[columns_to_standardize].agg(['mean', 'std'])

# z-score normalization
for col in columns_to_standardize:
    mu = mu_std.loc['mean', col]
    std = mu_std.loc['std', col]
    # new columns with normalized values
    song_df[f'NORM_{col}'] = (song_df[col] - mu) / std

# save new columns to our song_features table
db_connection = sqlite3.connect('dbs/song_features.db')
song_df.to_sql('song_features', db_connection, if_exists='replace', index=False)

db_connection.close()

In [ ]:
# ELBOW METHOD RESULTS FOR K-MEANS CLUSTERING
# From the results, k = 4 is the optimal number of clusters.

conn = sqlite3.connect('dbs/song_features.db')
song_df = pd.read_sql_query("SELECT * from song_features", conn)
conn.close()

clustered_features = song_df[['NORM_in_number_of_playlists', 'NORM_avg_playlist_followers', 'NORM_avg_position_in_playlist']]

inertias = []
k_range = range(3, 11)
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(clustered_features)
    inertias.append(kmeans.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(k_range, inertias, marker='o', linestyle='--')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')
plt.xticks(k_range)
plt.grid(True)
plt.show()

In [ ]:
# PERFORMS K-MEANS CLUSTERING ON SONG FEATURES
conn = sqlite3.connect('dbs/song_features.db')
song_df = pd.read_sql_query("SELECT * from song_features", conn)
conn.close()

# -- Chosen K value -- 4 was the best choice from the elbow method and was expected in our initial plan.
k = 4
# --------------------

clustered_features = song_df[['NORM_in_number_of_playlists', 'NORM_avg_playlist_followers', 'NORM_avg_position_in_playlist']]
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
kmeans.fit(clustered_features)

song_df['cluster'] = kmeans.labels_
features_to_analyze = ['in_number_of_playlists', 'avg_playlist_followers', 'avg_position_in_playlist', 'cluster']
analysis_df = song_df[features_to_analyze]

cluster_summary = analysis_df.groupby('cluster').agg(['mean', 'size'])
print(cluster_summary)

# -- Save to table --
conn = sqlite3.connect('dbs/song_features.db')
song_df.to_sql('song_features', conn, if_exists='replace', index=False)
conn.close()

# From the output results, each cluster was manually labeled as follows:
# Cluster 0: Generic
# Cluster 1: Niche
# Cluster 2: Mainstream
# Cluster 3: Curated

In [ ]:
# CLUSTERING VISUALIZATION — CREATES A ROTATING .gif FILE THAT SHOWS THE 3D VISUALIZATION OF THE CLUSTERING RESULTS.
# Note: This takes about ~30 minutes to run. So, it is commented out but left in.

# conn = sqlite3.connect('dbs/song_features.db')
# songs_df = pd.read_sql_query("SELECT * FROM song_features", conn)
# conn.close()
# fig = plt.figure(figsize=(10, 8))
# ax = fig.add_subplot(111, projection='3d')
# scatter = ax.scatter(songs_df['NORM_in_number_of_playlists'], 
#                      songs_df['NORM_avg_playlist_followers'], 
#                      songs_df['NORM_avg_position_in_playlist'], 
#                      c=songs_df['cluster'], 
#                      cmap='viridis', 
#                      s=40, alpha=0.7)

# ax.set_xlabel('NORM_playlist_count')
# ax.set_ylabel('NORM_avg_followers')
# ax.set_zlabel('NORM_avg_position')
# ax.set_title('3D Cluster Rotation')

# def rotate(angle):
#     ax.view_init(elev=20, azim=angle)

# rot_animation = animation.FuncAnimation(fig, rotate, frames=np.arange(0, 360, 10), interval=200)
# rot_animation.save('3d_clusters.gif', dpi=50, writer='pillow')

In [ ]:
# PERFORMS TF-IDF TO EXTRACT TOP 100 THEME/GENRES FROM PLAYLIST TITLES, CALCULATING A STAPLE SCORE AND ADDING THESE TO A .pkl FILE
nltk.download('stopwords')
stopwords_set = set(stopwords.words('english'))

def clean_title(title):
    title = title.lower()
    title = re.sub(r'[^a-z0-9\s]', '', title)  # Remove punctuation
    tokens = title.split()
    tokens = [t for t in tokens if t not in stopwords_set]
    return ' '.join(tokens)

conn = sqlite3.connect('dbs/playlists.db')
df_titles = pd.read_sql_query("SELECT pid, name FROM playlists", conn)
conn.close()

df_titles["tokens"] = df_titles["name"].fillna("").apply(clean_title)

# Create TF-IDF matrix
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df_titles["tokens"])

top_n = 100 # Number of top themes to consider

term_frequencies = np.asarray(tfidf_matrix.sum(axis=0)).flatten()
top_indices = np.argsort(-term_frequencies)[:top_n]
top_terms = vectorizer.get_feature_names_out()[top_indices]

binary_matrix = (tfidf_matrix[:, top_indices] > 0).astype(int)

playlist_theme_df = pd.DataFrame.sparse.from_spmatrix(binary_matrix, columns=top_terms, index=df_titles["pid"])

def calc_staple_scores(theme_df, tracks_df):
    pid_to_index = {pid: idx for idx, pid in enumerate(theme_df.index)} # mapping playlist ID to row index
    theme_matrix = sp.csr_matrix(theme_df.sparse.to_coo().astype(int)) # converting to SPARSE matrix, as playlists will likely have 1 or few themes

    unique_tracks = tracks_df['track_uri'].unique()
    track_to_index = {track: idx for idx, track in enumerate(unique_tracks)} # mapping track URI to column index

    #playlist -> song sparse matrix
    row_indices = tracks_df['pid'].map(pid_to_index).values # from each track-row, map to its playlist index it belongs to
    col_indices = tracks_df['track_uri'].map(track_to_index).values # from each track-row, map to its song index it is

    data = np.ones(len(tracks_df), dtype=int) # creating binary matrix; 1s mean song is in playlist, 0s mean it is not
    playlist_song_matrix = sp.csr_matrix((data, (row_indices, col_indices)), shape=(len(pid_to_index), len(track_to_index))) # sparse matrix of which songs are in which playlists

    theme_song_counts = theme_matrix.T @ playlist_song_matrix # themes (dot product) songs = for each theme, how many playlists with that theme have each song
    playlists_per_theme = theme_matrix.sum(axis=0).A1 # Count how many playlists have each theme
    playlists_per_theme[playlists_per_theme == 0] = 1  # Avoid division by zero

    D_inv = sp.diags(1 / playlists_per_theme)  # Create a diagonal matrix for normalization
    staple_scores = D_inv.dot(theme_song_counts)  # Normalized; dot-product of the inverse is equivalent to divide total playlists for that theme to get fraction staple-score

    staple_scores_transposed = staple_scores.T

    song_theme_staple_df = pd.DataFrame(
        staple_scores_transposed.toarray(),
        index=unique_tracks,
        columns=playlist_theme_df.columns)
    
    return song_theme_staple_df

conn = sqlite3.connect('dbs/playlists.db')
tracks_df = pd.read_sql_query("SELECT pid, track_uri FROM playlist_track", conn)
conn.close()

staple_scores_df = calc_staple_scores(playlist_theme_df, tracks_df)
staple_scores_df.index.name = 'track_uri'
staple_scores_df = staple_scores_df.reset_index()

staple_scores_df.to_pickle('dbs/staple_scores.pkl') # Pickle is much faster, but not human-readable

In [ ]:
# MERGES THE ENGINEERED STAPLE SCORES WITH THE song_features TABLE AND GENERATES role_annotations FOR EACH SONG

conn = sqlite3.connect('dbs/song_features.db')
song_features_df = pd.read_sql_query("SELECT * FROM song_features", conn)
conn.close()

staple_scores_df = pd.read_pickle('dbs/staple_scores.pkl')
print(staple_scores_df.columns)
#staple_scores_df = staple_scores_df.reset_index().rename(columns={'index': 'track_uri'})

merged_df = pd.merge(
    song_features_df,
    staple_scores_df,
    how='left',
    on='track_uri'
)

theme_columns = [col for col in staple_scores_df.columns if col != 'track_uri']
merged_df[theme_columns] = merged_df[theme_columns].fillna(0)

cluster_labels = {
    0: 'Generic',
    1: 'Niche',
    2: 'Mainstream',
    3: 'Curated'}

#theme_names_list = staple_scores_df.columns.tolist()
#staple_score_columns = [col for col in merged_df.columns if col in theme_names_list]
#merged_df[staple_score_columns] = merged_df[staple_score_columns].apply(pd.to_numeric, errors='coerce')

# merged_df['role_annotation'] = merged_df['cluster'].map(cluster_labels) + ' ' + merged_df[staple_score_columns].idxmax(axis=1).str.title() # cluster label + highest staple score theme
# missing_theme = (merged_df[staple_score_columns].max(axis=1) == 0) | (merged_df[staple_score_columns].isna().all(axis=1)) # Identify rows with no staple scores
# merged_df.loc[missing_theme, 'role_annotation'] = merged_df.loc[missing_theme, 'cluster'].map(cluster_labels) # Assign generic role (no theme label) for missing staple scores

merged_df['role_annotation'] = merged_df['cluster'].map(cluster_labels) + ' ' + merged_df[theme_columns].idxmax(axis=1).str.title()

missing_theme = (merged_df[theme_columns].max(axis=1) == 0) | (merged_df[theme_columns].isna().all(axis=1))

merged_df.loc[missing_theme, 'role_annotation'] = merged_df.loc[missing_theme, 'cluster'].map(cluster_labels)

conn = sqlite3.connect('dbs/song_features.db')
cols_to_keep = ['track_uri', 'track_name', 'artist_name', 'in_number_of_playlists', 'avg_playlist_followers', 'avg_position_in_playlist',
                 'NORM_in_number_of_playlists', 'NORM_avg_playlist_followers', 'NORM_avg_position_in_playlist', 'cluster', 'role_annotation']
final_df = merged_df[cols_to_keep].copy()

if final_df['track_uri'].isnull().any():
    raise ValueError('Null track_uri found before writing to DB')

final_df.to_sql('temp_song_features', conn, if_exists='replace', index=False)
cursor = conn.cursor()

try:
    cursor.execute(f"DROP TABLE IF EXISTS song_features")
    conn.commit()
    cursor.execute("ALTER TABLE temp_song_features RENAME TO song_features")
    conn.commit()
    print("successfully updated song_features table with role annotations")
except sqlite3.Error as e:
    print(f"Error updating table: {e}")

conn.close()

In [ ]:
# SUPERVISED LEARNING; TRAINS A DECISION TREE CLASSIFIER WITH A One vs. All STRATEGY

conn = sqlite3.connect('dbs/song_features.db')
song_features_df = pd.read_sql_query("SELECT * FROM song_features", conn)
conn.close()

features = ['NORM_in_number_of_playlists', 'NORM_avg_playlist_followers', 'NORM_avg_position_in_playlist']
X = song_features_df[features]
Y = song_features_df['cluster']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

clusters = Y.unique()
classifiers = {}
predictions = pd.DataFrame(index=X_test.index)

for cluster in clusters:
    Y_train_bin = (Y_train == cluster).astype(int)
    clf = DecisionTreeClassifier(class_weight='balanced', random_state=42)
    clf.fit(X_train, Y_train_bin)
    classifiers[cluster] = clf

    prediction_probability = clf.predict_proba(X_test)[:, 1]
    predictions[cluster] = prediction_probability

final_predictions = predictions.idxmax(axis=1)

# Model evaluation
print("Classification Report:")
print(classification_report(Y_test, final_predictions))
print("Confusion Matrix:")
print(confusion_matrix(Y_test, final_predictions))

In [ ]:
# CALCULATES A SILHOUETTE SCORE FOR MEASURING THE QUALITY OF THE CLUSTERING
sampled_df = song_df.sample(n=100000, random_state=42)

features = sampled_df[['NORM_in_number_of_playlists', 
                       'NORM_avg_playlist_followers', 
                       'NORM_avg_position_in_playlist']]
labels = sampled_df['cluster']

score = silhouette_score(features, labels)
print(f"Silhouette Score (sampled): {score:.4f}")